# 01 - Data Download
This notebook will be used to download the various datasets needed for the project. That includes:

| Dataset | Use | Code Complete? | Downloaded? |
| --- | --- | --- | --- |
| USGS/other federal boundary holder | site boundary that's used to select downloaded data | no | no |
| USGS DEM | digital elevation model used for SWE training | no | no |
| SNOTEL | timeseries used for SWE training | no | no |
| PRISM climate data | gridded reanalysis data used for SWE training | no | no |
| MACAv2 climate data | gridded future data used for SWE prediction | no | no |

## Step 1: Import Libraries

In [1]:
# Library management
import os
import pathlib

# Downloading
from tqdm.notebook import tqdm # progress bar
import requests # for SNOTEL API access
import zipfile

# Data Management
import pandas as pd
import xarray as xr

# Geospatial Management
import geopandas as gpd
from pygeohydro import WBD # site boundary based on watersheds
#import rasterio
import rioxarray as rxr

# Plotting
import matplotlib
import matplotlib.pyplot as plt
import holoviews as hv
from shapely.geometry import box

In [ ]:
# You may need to install pygeohydro 

# if so, uncomment and run this block to install the 
# package to your current kernel

# %pip install pygeohydro

   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ------------------------------------ --- 3.7/4.0 MB 18.2 MB/s eta 0:00:01
   ---------------------------------------- 4.0/4.0 MB 16.1 MB/s  0:00:00

   ----- ----------------------------------  2/15 [h5netcdf]
   ---------------- -----------------------  6/15 [owslib]
   ------------------ ---------------------  7/15 [aiohttp-client-cache]
   ------------------------------------- -- 14/15 [pygeohydro]
   ---------------------------------------- 15/15 [pygeohydro]

Note: you may need to restart the kernel to use updated packages.


## Step 2: Site Selection
In this step, I download HUC8 watershed boundaries to use as the bounding box for the project.

I've selected HUC8 watershed boundaries that will capture the following ranges:

**FILL IN**

**REASONING FOR HUC8 boundaries**

documentation on pygeohydro: https://docs.hyriver.io/readme/pygeohydro.html

Citation:
@article{Chegini_2021,
    author = {Chegini, Taher and Li, Hong-Yi and Leung, L. Ruby},
    doi = {10.21105/joss.03175},
    journal = {Journal of Open Source Software},
    month = {10},
    number = {66},
    pages = {1--3},
    title = {{HyRiver: Hydroclimate Data Retriever}},
    volume = {6},
    year = {2021}
}

### 2a: Download

In [2]:
# ── Step 1: Pull HUC8 boundaries for SW Montana bounding box ──────────────────
# (west, south, east, north) — generous box, we'll filter down
bbox = (-114.5, 44.3, -109.5, 47.0)

wbd = WBD("huc8")
huc8_all = wbd.bygeom(box(*bbox), geo_crs=4326)

print(f"Total HUC8s in bounding box: {len(huc8_all)}")
print(huc8_all[["huc8", "name"]].to_string())

# ── Step 2: Filter to Upper Missouri headwaters (HUC4 = 1002) ─────────────────
huc8_missouri = huc8_all[huc8_all["huc8"].str.startswith("1002")].copy()

print(f"\nUpper Missouri HUC8s: {len(huc8_missouri)}")
print(huc8_missouri[["huc8", "name"]].to_string())

# ── Step 3: Visualize — inspect before committing to anything ─────────────────
fig, ax = plt.subplots(figsize=(12, 9))
huc8_missouri.plot(
    ax=ax,
    alpha=0.4,
    edgecolor="black",
    linewidth=0.8,
    column="name",
    cmap="tab20",
    legend=False
)
# Label each watershed with name and code
for _, row in huc8_missouri.iterrows():
    centroid = row.geometry.centroid
    ax.annotate(
        f"{row['name']}\n({row['huc8']})",
        xy=(centroid.x, centroid.y),
        ha="center",
        fontsize=7,
        color="black"
    )
ax.set_title("Upper Missouri HUC8 Watersheds — SW Montana", fontsize=13)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()
plt.savefig("figures/huc8_study_area.png", dpi=150)
plt.show()

ClientConnectorDNSError: Cannot connect to host hydro.nationalmap.gov:443 ssl:default [Could not contact DNS servers]

### 2b: Spatial Join
need to join and dissolve

### 2c: Plot Boundary

## Step 3: Access SNOTEL Data

API Documentation: https://wcc.sc.egov.usda.gov/awdbRestApi/v3/api-docs
Interactive API demo: https://wcc.sc.egov.usda.gov/awdbRestApi/swagger-ui/index.html
GitHub Demo repo: https://github.com/nrcs-nwcc/iow_awdb_rest_api_demo

### 3a: Download station metadata

**I can search by HUC codes. Get a list of HUC codes from step above.**

This step gets the metadata for each station, which I can then filter using the watershed boundary.

In [ ]:
### From Claude:
# this downloads station metadata for all the stations in MT

# Get station metadata for Montana
url = "https://wcc.sc.egov.usda.gov/awdbRestApi/services/v1/stations"
params = {
    "stateCode": "MT",
    "networkCd": "SNTL",  # SNOTEL network code
    "elementCd": "SWE"    # filter for stations measuring SWE
}
response = requests.get(url, params=params)
stations = response.json()

### 3b: Filter stations

#### 3c: Test download func for one site

#### 3d: Download all sites

### 3?: SNOTEL Site Map
Make a map of SNOTEL sites to (a) confirm site locations and (b) provide a plot of where sites are located.

## Step 4: Access DEM

## Step 5: Access PRISM Data

## Step 6: Access MACAv2 Data

### 6a: Data dir

In [ ]:
# make directory for climate data

clim_dir = os.path.join(data_dir, 'model_data')
os.makedirs(clim_dir, exist_ok = True)

### 6b: Define functions

Functions to help process climate data.

In [ ]:
# convert temp from Kelvin to F function
def convert_K_to_F(da):
    '''
    Converts an xarray DataArray from Kelvin to Fahrenheit 

    Args:
    da (DataArray):
        DA of temperature values
    '''

    # Perform conversion pixel-wise
    da_f = (da * 1.8) - 459.67
    
    return da_f

In [ ]:
# convert longitude
def convert_longitude(longitude):

    '''
    This function cleans up datasets read from NetCDF files 
    in order to work with the other datasets used in this notebook.
    It converts lon from 0 to 360 to -180 to 180 

    Args:
    =====
    longitude (DataArray):
        DA from larger Dataset that contains longitude data

    Returns:
    ========
    longitude (DataArray)
    '''

    return longitude - 360 if longitude > 180 else longitude 

In [ ]:
# Function to download model data and process it
# we will need to update this for the summer class - currently this function takes an annual mean.

def download_and_process_maca_data(
        site_gdf, site_name,
        models, rcp_value,
        variable, year_range,
        maca_dir):
    '''
    Downloads and processes MACA data for a given site.
    Processing includes clipping bounds and rewriting longitude values
    
    Args:
    =====
    site_gdf (geopandas.GeoDataFrame):
        Site boundary, used for clipping
    site_name (str):
        Name of site
    models (list):
        List of models to programmatically download
    rcp_value (str):
        RCP value for MACA url
    variable (str):
        Variable of interest
    year_range (str):
        Range of years to be downloaded
    maca_dir (str):
        Path for data to be saved
    '''

    # Initialize results list
    results = []

    # count total steps for progress bar
    total_steps = len(models) * len(year_range)

    # download and process files
    with tqdm(total=total_steps, desc=f"Processing {site_name}") as pbar:
        # loop through models
        for model in models:

            # hold onto each time chunk for aggregation at the end
            model_chunks = []
            # loop through year chunks
            for years in year_range:

                # Programmatically determine file path and download url
                file_name = f'maca_{model}_{site_name}_{rcp_value}_{years}_CONUS_monthly.nc'
                maca_path = os.path.join(maca_dir, file_name)
                maca_url = (
                    f'http://thredds.northwestknowledge.net:8080/thredds/dodsC/MACAV2/{model}/'
                    f'macav2metdata_{variable}_{model}_r1i1p1_{rcp_value}_{years}_CONUS_monthly.nc'
                )
                

                # Download or open file
                if not os.path.exists(maca_path):
                    print(f"{model}, {years}, {rcp_value} downloaded successfully!")
                    maca_ds = xr.open_dataset(maca_url, mask_and_scale = True).squeeze()
                    maca_ds.to_netcdf(maca_path)
                else:
                    print(f"{model}, {years}, {rcp_value} already exists")
                    maca_ds = xr.open_dataset(maca_path, mask_and_scale = True).squeeze()

                # Convert longitude
                # reassign lon
                maca_ds = maca_ds.assign_coords(
                    lon = ('lon', [convert_longitude(l) for l in maca_ds.lon.values])
                )

                # set crs
                maca_ds = maca_ds.rio.write_crs('EPSG:4326')
                
                # set spatial dimension to define da as spatial
                maca_ds = maca_ds.rio.set_spatial_dims(
                    x_dim = 'lon',
                    y_dim = 'lat'
                )

                # set site bounds
                site_rpj = site_gdf.to_crs(maca_ds.rio.crs)
                site_bounds = site_rpj.total_bounds

                # write no data as NaN
                maca_ds['air_temperature'].rio.write_nodata(np.nan, inplace=True)

                # crop to bounding box
                maca_ds_crop = maca_ds.rio.clip_box(*site_bounds)
                
                # Append this specific 5-year chunk to the model list
                model_chunks.append(maca_ds_crop)

                # Update progress bar
                pbar.update(1)

            # Combine all year chunks for this model into one Dataset
            combined_model_ds = xr.concat(model_chunks, dim='time')
            
            # Convert temperature
            combined_model_ds['air_temperature'] = convert_K_to_F(combined_model_ds['air_temperature'])

            # Mask No Data values for mean
            valid_data = combined_model_ds.where(combined_model_ds > -100)

            # Calculate the mean across the time dimension
            mean_model_ds = valid_data.mean(dim='time')

            # convert to DataArray
            mean_model_da = mean_model_ds['air_temperature']

            # rewrite NaN values
            mean_model_da.rio.write_nodata(np.nan, inplace=True)
            
            # store data
            results.append({
                'site_name': site_name,
                'model': model,
                'rcp_value': rcp_value,
                'variable': variable,
            #    'date_range': years,
                'data': mean_model_da
            })
            
    return results

### 6c: Model Selection
Do I need this section? TBD

1. Get total bounds for each site to input into MACA [scatterplot viewer](https://climate.northwestknowledge.net/MACA/vis_scatterplot.php). 
2. Visualize ensemble spread. We used the mid-century time period, and plotted summer temperature change against winter precipitation change.
3. Find models that capture 4 scenarios: Hot-Wet, Hot-Dry, Cold-Wet, Cold-Dry. We selected models from the RCP4.5 pathway.

### 6d: Download Model Data

In [ ]:
# define year range and model lists

# define year chunks
### REVISE THIS
historic_range = ['1970_1974', '1975_1979', '1980_1984', 
                  '1985_1989', '1990_1994', '1995_1999']
future_range = ['2036_2040', '2041_2045', '2046_2050', 
                '2051_2055', '2056_2060', '2061_2065']

# define models
models = {
    "site_name": [
        "model_name1", "model_name2", "etc"],
}

In [ ]:
# set site names and gdfs
sites = {
    # fill in
}

# set scenarios
scenarios = [
    {'name': 'historical', 'range': historic_range},
    {'name': 'rcp45', 'range': future_range}
]

# initialize list of all the results
model_results = {}

# loop through sites and scenarios
for site_name, gdf in sites.items():
    for scn in scenarios:
        # Key for results
        result_key = f"{site_name.lower()}_{scn['name']}"

        # Print status update
        print(f"\n--- Running {site_name} | {scn['name']} ---")

        model_results[result_key] = download_and_process_maca_data(
            site_gdf = gdf,
            site_name = site_name,
            models = models[site_name.lower()],
            rcp_value = scn['name'],
            # downloading max temp here, but can change to other variables
            variable = 'tasmax',
            year_range = scn['range'],
            maca_dir = clim_dir
        )

### 6e: Save Model Data

THIS SECTION NEEDS TO BE COMPLETED.